# GrandNexus v7.6.0-rc — the living bench in real language, phase B (logbook 82)
The anchored LLM translator reads paraphrased facts ("Bob plays chess.", "Alice moved to Paris.") session by session, with
a **shared lexicon** given by the memory, and its programs feed the structured memory. Controls: the grammar pathway, the
translator without lexicon, the translator reading everything at once. Sandbox (no claim).
**About 200 calls, about $1.2, 15-25 minutes.** Cell 4 prints the exact count before anything is spent.

Expected in cell 5: two tables (levels 1 and 2); `llm_lexicon` far above `grammar` (about 25%) and above `llm_nolexicon`.
Run the cells in order. **If a cell prints STOP, do not run the next ones: send the output.** If the Colab assistant offers to
rewrite a cell, decline. Send back the output of cells 4, 5 and 6 and the zip named in cell 6.

## 1. Get exactly v7.6.0-rc

In [ ]:
# 1. Get EXACTLY GrandNexus v7.6.0-rc (never another zip; stop if it is not found)
import os, glob, fnmatch, subprocess, zipfile, shutil, time
VERSION, ZIP_PATTERN = 'v7.6.0-rc', 'GrandNexus_v7.6.0-rc*.zip'
CONTENT, WORK = '/content', '/content/GrandNexus'
MYDRIVE = '/content/drive/MyDrive'

def gn_version(path):
    """(git describe, has the story-aware bench) of a GrandNexus folder, or (None, False)."""
    if not os.path.isdir(os.path.join(path, '.git')):
        return None, False
    d = subprocess.run('git describe --tags', shell=True, cwd=path, capture_output=True, text=True).stdout.strip()
    bench = os.path.join(path, 'tools', 'bench_learning.py')
    ok = all(os.path.exists(os.path.join(path, 'tools', f)) for f in ('colab_save.py', 'living_llm.py', 'living_language.py')) and \
         os.path.exists(bench) and '--story-mode' in open(bench, encoding='utf-8').read()
    return d, ok

try:
    from google.colab import drive; drive.mount(os.path.join(CONTENT, 'drive'))
except Exception as e:
    print('Drive not mounted:', e)

# the release zip, newest first: My Drive/GrandNexus and My Drive first, then anywhere in My Drive (3 levels deep);
# any other GrandNexus zip (older versions, session saves) is ignored
found = glob.glob(os.path.join(MYDRIVE, 'GrandNexus', ZIP_PATTERN)) + glob.glob(os.path.join(MYDRIVE, ZIP_PATTERN))
if not found:
    for root, dirs, names in os.walk(MYDRIVE):
        if root[len(MYDRIVE):].count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(root, n) for n in names if fnmatch.fnmatch(n, ZIP_PATTERN)]
found.sort(key=os.path.getmtime)
if found:
    source = found[-1]
else:
    print(f'{ZIP_PATTERN} not found in My Drive: upload it now (any other zip is refused).')
    try:
        from google.colab import files
        up = next(iter(files.upload()))
    except Exception as e:
        raise SystemExit(f'STOP: no {ZIP_PATTERN} on the Drive and no upload possible ({e}).')
    if not fnmatch.fnmatch(os.path.basename(up), ZIP_PATTERN):
        raise SystemExit(f'STOP: {up} is not {ZIP_PATTERN}.')
    source = os.path.join(CONTENT, up)
print('release zip:', source)

# an existing /content/GrandNexus of another version is moved aside, never reused, never deleted
d, ok = gn_version(WORK)
if os.path.exists(WORK) and not (d or '').startswith(VERSION):
    aside = WORK + '_aside_' + time.strftime('%Y%m%d_%H%M%S')
    shutil.move(WORK, aside); print(f'existing {WORK} ({d}) moved aside to {aside}')
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist() if n.strip('/')}
    z.extractall(CONTENT if top == {'GrandNexus'} else WORK)
    print('extracted')
subprocess.run('git checkout -q main', shell=True, cwd=WORK)
d, ok = gn_version(WORK)
print('git describe:', d, '| v7.6.0-rc tools present:', ok)
print(subprocess.run('git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
if not ((d or '').startswith(VERSION) and ok):
    raise SystemExit(f'STOP: {WORK} is {d}, not {VERSION}. Do not run the next cells.')
print(f'OK: {VERSION}')


## 2. Packages, key, pre-flight (1 call)

In [ ]:
# 2. Packages, key, pre-flight check (1 call, through the translator itself: the cached system prompt must be accepted)
import subprocess, sys, os
WORK = '/content/GrandNexus'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic', 'clingo', 'sympy', 'networkx', 'psutil'])
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, WORK)
from grandnexus.llm_interface.translator_v5 import Translator, SYSTEM_TRANSLATE
tr = Translator(model='claude-sonnet-5-5')
reply = tr._call(SYSTEM_TRANSLATE, 'TEXT:\nAll dogs bark. Rex is a dog.\n\nQUESTION (or statement to verify):\nDoes Rex bark?', 'preflight')
r = tr.receipts[-1]
print('reply received:', bool(reply.strip()), '| id:', r.id, '| model:', r.model)
print(f'usage: input {r.tokens_in} | output {r.tokens_out} | cache write {r.cache_write} | cache read {r.cache_read}')
print('the system prompt went into the cache' if (r.cache_write or r.cache_read) else 'WARNING: no cache write or read on this call (send this output)')


## 3. Offline tests (0 calls)

In [ ]:
# 3. Offline tests (0 calls)
import subprocess, sys, glob, os
WORK = '/content/GrandNexus'
bad = []
for t in sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'test_*.py'))):
    r = subprocess.run([sys.executable, '-W', 'ignore', t], cwd=WORK, capture_output=True, text=True)
    last = (r.stdout.strip().splitlines() or ['(no output)'])[-1]
    ok = r.returncode == 0 and not last.startswith('FAIL') and 'Traceback' not in r.stderr
    print(f"{'ok ' if ok else 'BAD'} {os.path.basename(t):40} {last[:70]}")
    if not ok:
        bad.append(t)
if bad:
    raise SystemExit(f'STOP: {len(bad)} offline test(s) failed; send this output.')


## 4. Dry run (0 calls)

In [ ]:
# 4. What the run will cost (0 calls)
import subprocess, sys
WORK = '/content/GrandNexus'
r = subprocess.run([sys.executable, '-W', 'ignore', 'tools/living_llm.py', '--seed', '11', '--worlds', '3', '--sessions', '12', '--levels', '1,2', '--concat-worlds', '1', '--dry-run'], cwd=WORK, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-300:])


## 5. Phase B

In [ ]:
# 5. Phase B (about 200 calls, about $1.2, 15-25 minutes; keep the tab open: there is no resume in this sandbox tool)
import subprocess, sys
WORK = '/content/GrandNexus'
p = subprocess.Popen([sys.executable, '-W', 'ignore', 'tools/living_llm.py', '--seed', '11', '--worlds', '3', '--sessions', '12', '--levels', '1,2', '--concat-worlds', '1'], cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line:
        sys.stdout.write(line)
p.wait(); print('exit code', p.returncode)


## 6. Save to Drive

In [ ]:
# 6. Save to Drive (the repository's own tool: it zips the repository it belongs to and checks the result is inside)
import subprocess, sys
WORK = '/content/GrandNexus'
r = subprocess.run([sys.executable, 'tools/colab_save.py', '--tag', 'phaseB', '--expect', 'tests_lab/resultats/living/living_llm_B_*.json'],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
